![BasQ banner](logo_cropped.png)

# <b>Ground-State Energy of a Molecule with VQE — Real Quantum Hardware</b>
**Author:** Benjamin Tirado  
**Created for:** BasQ Qiskit Fall Fest 2026 — Basque Quantum (BasQ)

<a id="goal"></a>
<div class="alert alert-block alert-success">

<b>Goal of this notebook:</b> Compute the <b>ground-state energy of the hydrogen molecule (H₂)</b> at its equilibrium bond distance using the <b>Variational Quantum Eigensolver (VQE)</b> running on a <b>real IBM Quantum device</b>. The chemistry workflow is identical to the simulator version — PySCF → Jordan-Wigner → UCCSD ansatz — but the execution layer is replaced with <code>qiskit_ibm_runtime</code>, and we add <b>hardware-aware transpilation</b> and <b>error suppression</b> to compensate for real device noise.
</div>

# <b>Table of Contents</b>

* [Background](#background)
* [Pre-requisites & Authentication](#prereq)
* [Defining the Molecule](#molecule)
* [Mapping to Qubits: Jordan-Wigner](#mapping)
* [The Ansatz: Hartree-Fock + UCCSD](#ansatz)
* [Hardware-Aware Transpilation](#transpile)
* [Running VQE on Hardware](#vqe)
* [Results and Evaluation](#res)
* [Moving forward](#move)
* [Useful resources](#use)

## <b>Background</b> <a id="background"></a>

One of the most natural applications of a quantum computer is simulating **quantum systems** themselves. In quantum chemistry, a central task is to find the **ground-state energy** of a molecule: the lowest possible energy of its electrons for a fixed arrangement of nuclei. This single number governs much of a molecule's chemistry, from its stability to the shape of its bonds.

Classically, the cost of solving the electronic Schrödinger equation exactly grows extremely fast with the number of electrons, quickly becoming intractable. This is where quantum computers offer a promising route: the state of a molecule can be encoded into qubits, and its energy estimated by preparing trial states and measuring them.

The **Variational Quantum Eigensolver (VQE)** is a hybrid quantum-classical algorithm built for exactly this task. It relies on the **variational principle**, which guarantees that for any trial state $|\psi(\vec{\theta})\rangle$,

$$
E(\vec{\theta}) = \frac{\langle \psi(\vec{\theta}) | \hat{H} | \psi(\vec{\theta}) \rangle}{\langle \psi(\vec{\theta}) | \psi(\vec{\theta}) \rangle} \;\geq\; E_0,
$$

where $\hat{H}$ is the molecular Hamiltonian and $E_0$ is the true ground-state energy. VQE exploits this by parameterizing the trial state with a quantum circuit (the **ansatz**) and using a **classical optimizer** to tune the parameters $\vec{\theta}$ until the measured energy is as low as possible.

### Running on real hardware vs. a simulator

On a **noiseless simulator** the algorithm converges cleanly to the exact answer. On **real hardware**, every gate introduces small errors, measurements are noisy, and the device's qubit connectivity forces extra SWAP gates that deepen the circuit further. The measured energy will therefore be *higher* than the true ground state — but with careful **transpilation**, **error suppression** (dynamical decoupling, gate twirling), and **error mitigation** (zero-noise extrapolation, measurement-error correction) we can push it close to chemical accuracy.

This notebook shows you how to make that transition from the simulator version step by step.

# <b>Pre-requisites & Authentication</b> <a id="prereq"></a>

Install all required packages:

In [ ]:
!pip install qiskit qiskit_aer qiskit_ibm_runtime qiskit[visualization] qiskit_nature pyscf qiskit_algorithms

### Authenticate with IBM Quantum

Go to [https://quantum.ibm.com](https://quantum.ibm.com), log in, and copy your API token from the dashboard.  
Paste it below and run the cell **once** — your credentials will be saved locally for future sessions.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Save your credentials (run once, then you can comment this out)
QiskitRuntimeService.save_account(
    token="<Your-API-Key>",  # ← replace with your 44-character API key
    overwrite=True,
)

service = QiskitRuntimeService()

### Select a backend

Choose a real device. The cell below picks the **least-busy** QPU with at least 5 qubits (our H₂ problem needs 4).  
Alternatively, uncomment the last line to target a specific machine (e.g. `ibm_basquecountry`).

In [ ]:
# Least-busy real device with ≥ 5 qubits
backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)

# Or target a specific device:
# backend = service.backend("ibm_basquecountry")

print(f"Using backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import SPSA

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import EstimatorV2 as Estimator, EstimatorOptions, Session, Batch

# <b>Defining the Molecule</b> <a id="molecule"></a>

This step is **identical** to the simulator notebook. We describe H₂ at its equilibrium bond distance of 0.735 Å using the minimal `sto3g` basis set. The `PySCFDriver` runs a classical Hartree-Fock calculation and returns an `ElectronicStructureProblem` containing the second-quantized Hamiltonian:

$$
\hat{H} = \sum_{pq} h_{pq}\, a_p^\dagger a_q + \tfrac{1}{2}\sum_{pqrs} h_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s
$$

> **Note:** PySCF is a classical library and runs on your laptop — it is not affected by which quantum backend you chose above.

In [ ]:
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

problem = driver.run()

print("Number of spatial orbitals:", problem.num_spatial_orbitals)
print("Number of particles (alpha, beta):", problem.num_particles)
print("Nuclear repulsion energy:", problem.nuclear_repulsion_energy)

# <b>Mapping to Qubits: Jordan-Wigner</b> <a id="mapping"></a>

We apply the **Jordan-Wigner** transformation to convert the fermionic Hamiltonian into a weighted sum of Pauli strings acting on 4 qubits:

$$
\hat{H} = \sum_j c_j \, \hat{P}_j, \qquad \hat{P}_j \in \{I, X, Y, Z\}^{\otimes 4}
$$

This Pauli-operator form is exactly what the `Estimator` primitive on the hardware can measure.

In [ ]:
mapper = JordanWignerMapper()

fermionic_op = problem.hamiltonian.second_q_op()
qubit_op = mapper.map(fermionic_op)

print("Number of qubits:", qubit_op.num_qubits)
print("Number of Pauli terms:", len(qubit_op))
print(qubit_op)

# <b>The Ansatz: Hartree-Fock + UCCSD</b> <a id="ansatz"></a>

We construct the same **UCCSD ansatz** initialized from the **Hartree-Fock** reference state as in the simulator notebook. For H₂ / STO-3G this yields 4 qubits and 3 variational parameters.

On hardware the UCCSD circuit will be **transpiled** to match the device's native gate set and qubit connectivity in the next section. Deep circuits are more sensitive to noise, so keeping the circuit as short as possible matters.

In [ ]:
hf_state = HartreeFock(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
)

ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=hf_state,
)

print("Number of qubits:", ansatz.num_qubits)
print("Number of variational parameters:", ansatz.num_parameters)
ansatz.decompose().draw("mpl", fold=-1)

# <b>Hardware-Aware Transpilation</b> <a id="transpile"></a>

Before a circuit can run on a real device, it must be **transpiled** to the device's:
- **Native gate set** (e.g. `ECR`, `SX`, `RZ`, `X` on IBM QPUs with heavy-hex topology)
- **Qubit connectivity** (not all pairs of qubits can interact directly; SWAP gates are inserted where needed)

We use `generate_preset_pass_manager` with `optimization_level=3` (the most aggressive gate reduction) to keep the circuit as shallow as possible, which reduces the accumulated noise.

We also apply the transpilation to the **observables** (the Pauli terms of the Hamiltonian) so that qubit indices match after layout assignment.

> **Tip:** The transpiled circuit will typically be *deeper* than the abstract UCCSD circuit because of SWAP routing. Printing its gate counts below helps you estimate the noise budget.

In [ ]:
# Transpile ansatz to the target backend
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_ansatz = pm.run(ansatz)

# Apply the same layout to the Hamiltonian observables
isa_qubit_op = qubit_op.apply_layout(isa_ansatz.layout)

print("Transpiled circuit depth:", isa_ansatz.depth())
print("Gate counts:", dict(isa_ansatz.count_ops()))
isa_ansatz.draw("mpl", idle_wires=False, fold=40)

# <b>Running VQE on Hardware</b> <a id="vqe"></a>

### Choosing an optimizer for hardware

On a simulator we used **SLSQP**, a gradient-based optimizer that is fast and accurate when there is no measurement noise. On hardware, every function evaluation is noisy, so gradient-based methods can chase noise rather than the true gradient.

**SPSA (Simultaneous Perturbation Stochastic Approximation)** is the standard choice for noisy hardware: it estimates gradients from only two circuit evaluations per step regardless of the number of parameters, and its stochastic nature makes it robust to shot noise.

### Error suppression with the Runtime Estimator

`EstimatorV2` from `qiskit_ibm_runtime` runs circuits on the real device and exposes `EstimatorOptions` for error suppression and mitigation:

| Option | Effect |
|--------|--------|
| `resilience_level=1` | Readout-error mitigation via a local approach |
| `dynamical_decoupling.enable=True` | Inserts DD pulse sequences in idle windows to suppress decoherence |
| `default_shots` | Number of shots per Pauli-group measurement (more = less shot noise, longer queue time) |

> **Note:** Each VQE iteration submits *one job per parameter update*. With `max_iterations=100` this means up to 100 jobs in the IBM Quantum queue. Start with a small number (e.g. 30–50) to prototype, then increase for a final run.

In [ ]:
# --- Exact classical reference (runs locally, no hardware needed) ---
numpy_solver = NumPyMinimumEigensolver()
exact_calc = GroundStateEigensolver(mapper, numpy_solver)
exact_result = exact_calc.solve(problem)
exact_energy = exact_result.total_energies[0]
print(f"Exact ground-state energy : {exact_energy:.6f} Ha")

In [ ]:
# --- Configure the hardware Estimator with error suppression ---
options = EstimatorOptions()
options.resilience_level = 1                          # readout-error mitigation
options.dynamical_decoupling.enable = True            # suppress decoherence in idle windows
options.dynamical_decoupling.sequence_type = "XY4"   # XY4 DD sequence
options.default_shots = 1024                          # shots per Pauli-group evaluation

hw_estimator = Estimator(mode=backend, options=options)

In [ ]:
import numpy as np
from qiskit_ibm_runtime import Batch, EstimatorV2 as Estimator


parameter_grid = [np.random.uniform(-np.pi/2, np.pi/2, isa_ansatz.num_parameters) for _ in range(30)]


pubs = []
for params in parameter_grid:

    bound_circuit = isa_ansatz.assign_parameters(params)

    pubs.append((bound_circuit, isa_qubit_op))

print(f"Send {len(pubs)} Circuits to real hardware on single Batch ..")


with Batch(backend=backend) as batch:

    hw_estimator = Estimator(mode=batch, options=options)


    job = hw_estimator.run(pubs)
    results = job.result()


electronic_energies = [float(res.data.evs) for res in results]

total_energies = [e_elec + problem.nuclear_repulsion_energy for e_elec in electronic_energies]

min_vqe_energy = min(total_energies)

print("\n--- Scan Results (Grid Search) ---")
print(f"Exact ground-state energy : {exact_energy:.6f} Ha")
print(f"Lowest VQE energy found   : {min_vqe_energy:.6f} Ha")
print(f"Absolute error            : {abs(min_vqe_energy - exact_energy):.2e} Ha")


# <b>Results and Evaluation</b> <a id="res"></a>

The plot below compares:
- **Hartree-Fock baseline** — the mean-field energy (no electron correlation), computed on the noiseless `StatevectorEstimator`.
- **VQE on hardware** — the energy returned by the real device after optimization.
- **Exact diagonalization** — the true ground-state energy from `NumPyMinimumEigensolver`.

On a noiseless simulator, VQE matches the exact answer almost perfectly. On real hardware, noise pushes the measured energy upward. The gap between the hardware result and the exact value reflects the combined effect of gate errors, readout noise, and finite shot counts. Error mitigation (`resilience_level=1`) and dynamical decoupling help close this gap.

> **Chemical accuracy** is defined as $|E_{\text{VQE}} - E_{\text{exact}}| < 1.6 \times 10^{-3}$ Ha ($\approx 1$ kcal/mol). Reaching it on current hardware for H₂ is challenging but possible with careful mitigation.

In [ ]:
# Hartree-Fock energy: evaluate Hamiltonian on the HF state using a noiseless estimator.
hf_circuit = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
_est = StatevectorEstimator()
_elec_hf = _est.run([(hf_circuit, qubit_op)]).result()[0].data.evs
hf_energy = float(_elec_hf) + problem.nuclear_repulsion_energy

labels   = ["Hartree-Fock\n(baseline)", "VQE\n(hardware)", "Exact\n(diagonalization)"]
energies = [hf_energy, min_vqe_energy, exact_energy]
colors   = ["#B0B0B0", "#ff7f0e", "#2ca02c"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, energies, color=colors)
plt.ylabel("Total energy (Hartree)")
plt.title(f"Ground-state energy of H$_2$ — {backend.name}")
plt.axhline(exact_energy, color="#2ca02c", ls="--", lw=1, alpha=0.7, label="Exact")
plt.axhline(exact_energy - 1.6e-3, color="gray", ls=":", lw=1, alpha=0.7,
            label="Chemical accuracy band")
plt.axhline(exact_energy + 1.6e-3, color="gray", ls=":", lw=1, alpha=0.7)

for bar, e in zip(bars, energies):
    plt.text(bar.get_x() + bar.get_width() / 2, e, f"{e:.4f}",
             ha="center", va="bottom", fontsize=10)

plt.legend()
plt.tight_layout()
plt.show()

error = abs(min_vqe_energy - exact_energy)
print(f"Absolute error from exact : {error:.2e} Ha")
print(f"Within chemical accuracy  : {error < 1.6e-3}")

# <b>Moving forward</b> <a id="move"></a>

You have just run VQE for H₂ on a real quantum device. The workflow — molecule → Jordan-Wigner → UCCSD → hardware Estimator with error suppression → SPSA — is the same template you would use for any electronic-structure problem on IBM Quantum hardware. The challenge levels below push this template further.

### <b>Beginner — a potential energy surface on hardware</b>
Compute the **potential energy surface (PES)** of **HeH⁺** as a function of bond distance, running each point on hardware. Compare the hardware curve to the exact classical PES. Where does noise hurt the most — near equilibrium, or at stretched geometries? Think about the shot budget: the PES requires many VQE runs, so you may need to reduce `max_iterations` per point.

### <b>Intermediate — reduce resources to fit the hardware</b>
UCCSD on a full spin-orbital mapping is already deep for current devices. Move to a **larger molecule** (LiH, BeH₂) and reduce the quantum resources to make the circuit feasible: qubit tapering via molecular symmetries, alternative mappings (parity, Bravyi-Kitaev), frozen-core approximations. Compare the accuracy you can reach on hardware before and after resource reduction.

### <b>Advanced — push noise mitigation further</b>
Explore stronger mitigation strategies available in `qiskit_ibm_runtime`:
- `resilience_level=2` — Zero-Noise Extrapolation (ZNE)
- `resilience_level=3` — Probabilistic Error Cancellation (PEC) *(more shots required)*
- Gate twirling (`twirling.enable_gates=True`)

How much does each level of mitigation improve the VQE energy? Is there a point of diminishing returns? Plot the hardware energy as a function of `resilience_level` alongside the exact reference.

### <b>Hardware-aware optimizer tuning</b>
SPSA is robust but slow to converge. Experiment with:
- Different `maxiter` values and their effect on convergence vs. queue time.
- Gradient-free alternatives such as `COBYLA` or `NELDER_MEAD`.
- Warm-starting hardware VQE from the optimal parameters found by the simulator run first.

# <b>Useful resources</b> <a id="use"></a>

### <b>Qiskit Nature (quantum chemistry)</b>

- [Qiskit Nature documentation](https://qiskit-community.github.io/qiskit-nature/)  
- [Ground-state solvers tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/03_ground_state_solvers.html)  
- [Qubit mappers tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/06_qubit_mappers.html)  
- [UCCSD ansatz documentation](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.circuit.library.UCCSD.html)  

### <b>Hardware execution, transpilation, and error mitigation</b>

- [Qiskit IBM Runtime documentation](https://quantum.cloud.ibm.com/docs/guides/tools-intro)  
- [Transpile with pass managers](https://quantum.cloud.ibm.com/docs/guides/transpile-with-pass-managers)  
- [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques)  
- [Combine error mitigation options with the Estimator primitive](https://quantum.cloud.ibm.com/docs/tutorials/combine-error-mitigation-techniques)  

### <b>Variational algorithms and optimizers</b>

- [VQE documentation](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.VQE.html)  
- [SPSA optimizer](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.optimizers.SPSA.html)  
- [Qiskit Algorithms optimizers](https://qiskit-community.github.io/qiskit-algorithms/apidocs/qiskit_algorithms.optimizers.html)  
- [Variational quantum algorithms course](https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/variational-quantum-algorithms)  

### <b>Background reading</b>

- [Kandala et al., *Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets*, Nature (2017)](https://www.nature.com/articles/nature23879)  
  The IBM Quantum paper that demonstrates VQE on real superconducting hardware.

## <b>Credits and license</b>

This notebook was written by **Benjamin Tirado** for the **BasQ Qiskit Fall Fest 2026**, organised by Basque Quantum (BasQ).

© 2026 Benjamin Tirado. The text, figures and explanations are released under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); the code is released under the
[Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0), the same license as Qiskit.

You are welcome to run, adapt and share this material — including as a starting point for your own
challenge submission — provided the attribution above is kept. If you reuse it publicly, please cite it as:

> B. Tirado, *Ground-State Energy of a Molecule with VQE — Real Quantum Hardware*, tutorial notebook, BasQ Qiskit Fall Fest, 2026.

Built with [Qiskit](https://www.ibm.com/quantum/qiskit), [Qiskit Nature](https://qiskit-community.github.io/qiskit-nature/)
and [PySCF](https://pyscf.org/), which remain the property of their respective authors and are used
under their own licenses.

*Questions, corrections or suggestions:* benjamin.tirado@ehu.eus